# 07 · Visual-token compression (stage S4)

Every method keeps a fraction `k` of the visual tokens before the decoder, through one call, `api.compress`, chosen by
`cfg["compressor"]`. This notebook checks each method on one GQA and one TextVQA dev item. Nothing here is a result:
it shows that the kept count is exact, that `k = 100%` changes nothing, and what the model answers.

| Method | Chooses tokens by | Code |
|---|---|---|
| `random`, `uniform`, `text_mean` | chance / even spacing / similarity to the mean question embedding | ours (controls) |
| `lowres` | nothing: a smaller image with about `k` times the tokens | config only (control) |
| `crisp_lite` | similarity to the question's nouns, plus a uniform fill | reimplemented from the CRISP paper |
| `rediprune` | relevance to the question and diversity | official |
| `divprune` | diversity only | official |
| `visionzip` | attention in the vision tower; the rest is merged | official |

Details and deviations: `docs/notes/compression_baselines.md`.

**How to run:** Run All, no edits, then paste the Step-4 output. One model copy is used. Needs spaCy with `en_core_web_sm`.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# GPU choice must precede CUDA init, which importing adaloop.api already does: the one import outside adaloop.api
from utils.gpus import select_for_notebook

CONFIG = "configs/mvp.yaml"
GPU_IDS, N_COPIES = select_for_notebook(CONFIG)

In [ ]:
import math
import os
import socket

In [ ]:
from adaloop import api

ROOT, cfg = api.setup_notebook("configs/mvp.yaml")
print("ROOT", ROOT, "| config_hash", api.config_hash(cfg), "| git", api.git_hash())

In [ ]:
import torch

IS_INARA = "inara" in socket.gethostname()
print("host", socket.gethostname(), "| on Inara:", IS_INARA)
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
for i in range(torch.cuda.device_count()):
    total_gib = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"GPU {i} | {torch.cuda.get_device_name(i)} | {total_gib:.0f} GiB")

In [ ]:
import pandas as pd
import spacy

In [ ]:
# Settings
METHODS = [m for m in api.METHODS if m != "none"]
KS = [0.25, 0.5]  # keep fractions checked here; 100% is checked separately
R = 3  # loop passes
NOUN_MODEL = cfg["compression"]["crisp_lite"]["spacy_model"]
METHODS, cfg["compression"]

### Step-1:- Load the model and the noun tagger

`crisp_lite` needs spaCy's small English model; loading it first fails fast if it is missing.

In [ ]:
print("spaCy", spacy.__version__, "|", [t.text for t in spacy.load(NOUN_MODEL)("What color is the car?") if t.pos_ == "NOUN"])

In [ ]:
print("[STAGE] load model")
vlm = api.load_models(cfg, N_COPIES)[0]
api.set_loops(vlm, R)

### Step-2:- Two items and the uncompressed reference

The reference is the model with every visual token and no compression step.

In [ ]:
items = [api.load_gqa(cfg, "dev", limit=1, root=ROOT)[0], api.load_textvqa(cfg, "dev", limit=1, root=ROOT)[0]]
[(it.benchmark, it.question, it.answers[:3]) for it in items]

In [ ]:
def run(it, method, k):
    """Compress one item with `method` at keep fraction `k`, then generate."""
    run_cfg = {**cfg, "compressor": method}
    inputs, info = api.compress(vlm, api.item_inputs(vlm, it, run_cfg), it, k, run_cfg)
    return api.generate_answers(vlm, [inputs], run_cfg)[0], info


print("[STAGE] reference")
reference = {}
for it in items:
    gen, info = run(it, "none", 1.0)
    reference[it.item_id] = gen
    print(f"{it.benchmark} | visual tokens {info['n_vis']} (grid {info['grid_hw']}) | raw {gen.raw_text!r} | "
          f"pred {api.read_answer(gen, cfg)!r} | gold {it.answers[:3]}")

### Step-3:- k = 100% changes nothing

At `k = 100%` a method keeps everything but still takes its packed path (vision tower run outside `generate`), so equal
token ids check the plumbing.

In [ ]:
print("[STAGE] k = 100%")
for method in METHODS:
    for it in items:
        gen, info = run(it, method, 1.0)
        same = torch.equal(gen.token_ids, reference[it.item_id].token_ids)
        assert same and info["n_vis"] == info["n_vis_full"], f"{method} {it.benchmark}: k=1 changed the output {gen.raw_text!r}"
    print(f"✓ {method}: same generated ids as no compression on both items")

### Step-4:- Each method at 25% and 50% (paste this output)

Per method, `k` and item: kept tokens of the full count, the raw output, the parsed answer and the references.
`lowres` cannot hit an exact count (the grid grows a row or column at a time).

In [ ]:
print("[STAGE] methods")
rows = []
for method in METHODS:
    for k in KS:
        for it in items:
            gen, info = run(it, method, k)
            pred = api.read_answer(gen, cfg)
            rows.append({"method": method, "k": k, "benchmark": it.benchmark, "n_vis": info["n_vis"],
                         "n_vis_full": info["n_vis_full"], "grid_hw": info["grid_hw"], "res_px": info["res_px"],
                         "pred": pred, "correct": api.score(it.benchmark, pred, it.answers),
                         "vit_s": info["vit_s"], "compression_s": info["compression_s"],
                         "compression_flops": info["compression_flops"], "fallback": info["fallback"]})
            note = f" | fallback {info['fallback']}" if info["fallback"] else ""
            print(f"{method:10s} k={k:.2f} {it.benchmark:7s} | kept {info['n_vis']}/{info['n_vis_full']}{note}")
            print(f"    raw {gen.raw_text!r} | pred {pred!r} | gold {it.answers[:3]}")
table = pd.DataFrame(rows)

In [ ]:
table[["method", "k", "benchmark", "n_vis", "n_vis_full", "res_px", "correct", "vit_s", "compression_s", "compression_flops"]]

In [ ]:
for r in table.itertuples():
    want = api.n_keep_for(r.n_vis_full, r.k)
    if r.method == "lowres":
        assert abs(r.n_vis - want) <= max(r.grid_hw), f"lowres {r.benchmark} k={r.k}: {r.n_vis} vs {want}, grid {r.grid_hw}"
    else:
        assert r.n_vis == want, f"{r.method} {r.benchmark} k={r.k}: kept {r.n_vis}, want {want}"
print(f"✓ kept counts: exact for every method at {KS} (lowres within one grid row)")

### Step-5:- Controller features, one item first (stage S4b)

For every cell (k × R) the controller reads, after the last pass: the last-token state `h_last`, how much it moved
(`delta_norm`), the attention on the visual tokens (`vis_attn_mass`), and from the first answer-token logits the
`entropy`, `top1_prob` and `kl_prev` (against the R-1 run). `kl_prev` and `delta_norm` are NaN at R = 1. The logit
features cost a coda call when read early; that cost is stored with them (`flops_readout`).

In [ ]:
FEATURE_METHOD = cfg["compressor"] if cfg["compressor"] != "none" else "text_mean"  # S5 picks the compressor later
feature_cfg = {**cfg, "compressor": FEATURE_METHOD}
FEATURE_DIR = ROOT / cfg["out_dir"] / "nb07" / "features" / "base" / api.config_hash(feature_cfg)
print(FEATURE_METHOD, "| k", cfg["k_grid"], "| R", cfg["R_grid"], "|", FEATURE_DIR)

In [ ]:
def feature_table(features):
    """One row per (k, R) cell, without the hidden state."""
    return pd.DataFrame([{"k": k, "R": p, **{n: v for n, v in f.items() if n != "h_last"}}
                         for (k, p), f in sorted(features.items())])


print("[STAGE] features, 1 item")
rows, features = api.run_item_cells(vlm, items[0], feature_cfg)
feature_table(features)

In [ ]:
cells = [(k, p) for k in cfg["k_grid"] for p in cfg["R_grid"]]
assert sorted(features) == sorted(cells) and len(rows) == len(cells), (sorted(features), len(rows))
for (k, p), f in features.items():
    assert torch.isfinite(f["h_last"]).all() and 0 <= f["vis_attn_mass"] <= 1 and 0 < f["top1_prob"] <= 1, (k, p)
    assert math.isnan(f["kl_prev"]) == (p == 1), (k, p, f["kl_prev"])
full_R3 = next(r for r in rows if r["k"] == 1.0 and r["R"] == 3)
api.set_loops(vlm, R)
assert full_R3["raw_output"] == reference[items[0].item_id].raw_text, "k=1, R=3 differs from the Step-2 reference"
print(f"✓ features: {len(cells)} cells, finite; the k=1 R=3 row answers like the reference")

#### Five items, saved

In [ ]:
N_FEATURE_ITEMS = 5
print("[STAGE] features, save")
feature_items = api.load_gqa(cfg, "dev", limit=3, root=ROOT) + api.load_textvqa(cfg, "dev", limit=2, root=ROOT)
for n, it in enumerate(feature_items, 1):
    rows, features = api.run_item_cells(vlm, it, feature_cfg)
    path = api.save_features(features, FEATURE_DIR, it.item_id)
    print(f"[{n}/{len(feature_items)}] {it.item_id} -> {path.name}")

In [ ]:
assert len(feature_items) == N_FEATURE_ITEMS
for it in feature_items:
    saved = api.load_features(FEATURE_DIR / f"{it.item_id}.pt")
    assert sorted(saved) == sorted(cells), (it.item_id, sorted(saved))
api.set_loops(vlm, R)
print(f"✓ features saved for {N_FEATURE_ITEMS} items x {len(cells)} cells")

### Step-6:- Which method? (stage S5)

Every method at k = 25% and 50%, plus the full model once, at R = 3 on dev. Per cell: accuracy, FLOPs (vision tower +
decoder + compression), latency and compression time, actual visual tokens.

Two flags in the next cell, both off: `RUN_SAMPLE` runs `sample_items` dev items per benchmark (look at it first),
`RUN_FULL_DEV` the whole dev split. Runs resume if stopped: finished rows are kept, whatever batch size or copy count
wrote them. Items are run `eval_batch_size` at a time on `n_copies` copies (checked just below); the vision tower runs
once per item. Batched rows have no per-item latency, so timing comes from a second pass: `latency_items` items per benchmark at
batch 1 on one copy (no contention), in its own folder.

In [ ]:
RUN_SAMPLE = False  # set True after the Step-4 output is pasted and OK'd
RUN_FULL_DEV = False  # set True after the sample table is OK'd
COMPARE_N = None if RUN_FULL_DEV else cfg["sample_items"]  # None: the whole dev split
COMPARE_DIR = ROOT / cfg["out_dir"] / "nb07" / ("compare_dev" if RUN_FULL_DEV else "compare_sample")
RUN_COMPARE = RUN_SAMPLE or RUN_FULL_DEV
cfg["compare"], len(api.comparison_cells(cfg)), COMPARE_DIR

#### Batched against batch 1

Before any batched run: 8 items (half GQA, half TextVQA) as one batch and each alone, for one pruning method and for
`visionzip` (it merges tokens) at k = 25%. Same compressed inputs, so only the batching differs. Any difference stops
the notebook: set `eval_batch_size: 1` in `configs/mvp.yaml` (it is not part of the config hash) and report it.

In [ ]:
N_BATCH_CHECK = 8  # items, half per benchmark
CHECK_CELLS = [("rediprune", 0.25), ("visionzip", 0.25)]
if RUN_COMPARE:
    print(f"[STAGE] batch of {N_BATCH_CHECK} against batch 1, R={cfg['compare']['R']}")
    check_items = api.load_gqa(cfg, "dev", limit=N_BATCH_CHECK // 2, root=ROOT) + \
        api.load_textvqa(cfg, "dev", limit=N_BATCH_CHECK // 2, root=ROOT)
    batch_check = pd.DataFrame(api.check_batched(vlm, check_items, cfg, CHECK_CELLS, cfg["compare"]["R"]))
batch_check if RUN_COMPARE else "skipped: both flags are False"

In [ ]:
if RUN_COMPARE:
    mismatch = batch_check[~batch_check.ids_equal]
    assert len(batch_check) == N_BATCH_CHECK * len(CHECK_CELLS)
    assert mismatch.empty, f"batching changed {len(mismatch)} of {len(batch_check)} outputs:\n{mismatch}"
    print(f"✓ batched = batch 1: same generated ids on {len(batch_check)} (item, method) pairs")
else:
    print("skipped: both flags are False")

In [ ]:
if RUN_COMPARE:
    print(f"[STAGE] comparison, batch {cfg['eval_batch_size']}, {N_COPIES} copies")
    vlms = api.load_models(cfg, N_COPIES)
    compare_items = {"gqa": api.load_gqa(cfg, "dev", limit=COMPARE_N, root=ROOT),
                     "textvqa": api.load_textvqa(cfg, "dev", limit=COMPARE_N, root=ROOT)}
    compare_rows = api.run_comparison(vlms, compare_items, cfg, COMPARE_DIR)
len(compare_rows) if RUN_COMPARE else "skipped: both flags are False"

#### Latency pass: batch 1, one copy

The same cells on `latency_items` dev items per benchmark, one item at a time on the first copy only. Only its timing
columns are used (medians per cell): `latency_s` = vision + compression + generate, `t_prefill_ms` (time to first token;
for `none` and `lowres` it includes the vision tower), `t_decode_ms`.

In [ ]:
if RUN_COMPARE:
    LATENCY_DIR = COMPARE_DIR.parent / f"{COMPARE_DIR.name}_latency"
    N_LATENCY = cfg["latency_items"]
    print(f"[STAGE] latency pass, {N_LATENCY} items per benchmark, batch 1, 1 copy | {LATENCY_DIR}")
    latency_items = {"gqa": api.load_gqa(cfg, "dev", limit=N_LATENCY, root=ROOT),
                     "textvqa": api.load_textvqa(cfg, "dev", limit=N_LATENCY, root=ROOT)}
    latency_rows = api.run_comparison(vlms[:1], latency_items, cfg, LATENCY_DIR, batch_size=1)
    results = pd.DataFrame(api.comparison_table(compare_rows, latency_rows))
results.sort_values(["benchmark", "k", "accuracy"], ascending=[True, True, False]) if RUN_COMPARE else "skipped: both flags are False"

In [ ]:
if RUN_COMPARE:
    n_items = {b: len(v) for b, v in compare_items.items()}
    assert len(results) == 3 * len(api.comparison_cells(cfg)), len(results)
    assert all(r.n == (sum(n_items.values()) if r.benchmark == "all" else n_items[r.benchmark]) for r in results.itertuples())
    assert all(r.latency_n == (2 * N_LATENCY if r.benchmark == "all" else N_LATENCY) for r in results.itertuples()), "latency pass incomplete"
    assert (results.latency_s > 0).all() and results.t_prefill_ms.notna().all()
    full = results[results.method == "none"].set_index("benchmark").gflops_total
    pruned = results[(results.k < 1) & (results.method != "none")]
    assert all(r.gflops_total < full[r.benchmark] for r in pruned.itertuples()), "a pruned cell costs more than the full model"
    print(f"✓ comparison: every cell has every item and its latency; every pruned cell is cheaper than the full model")
else:
    print("skipped: both flags are False")

#### Winner

Best accuracy over k = 25% and 50%; methods whose paired 95% bootstrap interval against it contains 0 count as tied,
and the cheapest of those wins. A suggestion: you freeze `compressor` in `configs/mvp.yaml`.
**Decide on the full-dev run only:** on the sample the intervals are wide, nearly everything ties, and the rule then
just picks the cheapest method (possibly a control).

In [ ]:
if RUN_COMPARE:
    winner, evidence = api.pick_winner(compare_rows, cfg)
    print("suggested compressor:", winner, "" if RUN_FULL_DEV else "(sample only: do not freeze it)")
pd.DataFrame(evidence) if RUN_COMPARE else "skipped: both flags are False"

In [ ]:
if RUN_COMPARE:
    api.save_table(results, "S5_compression" if RUN_FULL_DEV else "S5_compression_sample")
    print(f"## <date> · S5 · compression comparison (git {api.git_hash()}, dev n={n_items}, R={cfg['compare']['R']})")
    print(f"- Decision: compressor = {winner} because <reason>.")
    print(f"- Latency: median, batch 1, one copy, {N_LATENCY} items per benchmark.")
    print("| method | k | acc all | GFLOPs | latency s | prefill ms | decode ms | compression s | kept tokens |")
    print("|---|---|---|---|---|---|---|---|---|")
    for r in results[results.benchmark == "all"].sort_values(["k", "accuracy"], ascending=[True, False]).itertuples():
        print(f"| {r.method} | {r.k} | {r.accuracy:.1f} | {r.gflops_total:.1f} | {r.latency_s:.2f} | {r.t_prefill_ms:.0f} | {r.t_decode_ms:.0f} | {r.compression_s:.3f} | {r.n_vis:.0f} |")
else:
    print("skipped: both flags are False")

#### Figure F06

Accuracy against FLOPs (top) and latency (bottom) per method; the star is the full model.

In [ ]:
if RUN_COMPARE:
    api.set_paper_style()
    fig = api.plot_compression(results)
else:
    print("skipped: F06 needs a comparison run")

In [ ]:
if RUN_COMPARE:
    pdf_path, png_path = api.save_figure(fig, "F06_compression", final=False)
    assert pdf_path.is_file() and png_path.is_file()
    print("✓ F06 draft saved")
else:
    print("skipped: F06 not saved")